# How redundant predictors can change selection

A tree can use an exact copy of a signal wherever it would use the original.
Which one gets importance can depend on which column the learner uses.
Compare GrootCV with and without a collinearity filter on a controlled
fixture. The label depends on the signal, with no independent effect from
the copy.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from arfs.feature_selection import CollinearityThreshold, GrootCV

rng = np.random.default_rng(42)
X = pd.DataFrame(rng.normal(size=(320, 2)), columns=["signal", "noise"])
X["signal_copy"] = X["signal"]
y = 5 * X["signal"] + rng.normal(scale=0.1, size=len(X))
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

## Compare the two input designs

Fit the filter on training rows. On this fixture the collinearity filter
drops the first of the tied columns. The two GrootCV fits start with the
same input order and use the same settings.

In [2]:
filtered = CollinearityThreshold(threshold=0.95).fit(X_train)
designs = {
    "All features": (X_train, X_test),
    "Collinearity filtered": (filtered.transform(X_train), filtered.transform(X_test)),
}
rows = []
for name, (X_fit, X_eval) in designs.items():
    selector = GrootCV(
        objective="rmse",
        n_folds=3,
        n_iter=1,
        n_jobs=1,
        random_state=42,
        lgbm_params={"num_iterations": 40},
    ).fit(X_fit, y_train)
    rows.append((name, list(X_fit.columns), list(selector.transform(X_eval).columns)))

pd.DataFrame(rows, columns=["Design", "Candidates", "Selected columns"])

,Design,Candidates,Selected columns
0,All features,"[signal, noise, signal_copy]",[signal]
1,Collinearity filtered,"[noise, signal_copy]",[signal_copy]


The compared features can carry the same information under different
names. The choice of a representative will matter if measurement cost,
availability or missingness differs. This controlled comparison tells us
how selection changes on the fixture. Assess predictive impact separately
on held-out rows.